# Broader or narrower in *sound*? Audio embeddings, pseudo-genres and the same question

Notebook 02 compared the two streams through **artist IDs**: recommended listening had more distinct artists, but its artist
combinations were mixed. Artist IDs say nothing about what the music *sounds* like. Here we use the **128-dimensional audio
embeddings** (`embeddings.parquet`, from a CNN trained on spectrograms) as neutral ground truth.

| # | Analysis | Course link | Question |
|---|---|---|---|
| 1 | Do the embeddings carry signal? | validation | same artist/album closer than random? |
| 2 | Audio spread per user | diversity metric | are recommended tracks more spread out in sound? |
| 3 | Pseudo-genres: own spherical k-means | Week 6 | a map of the sound space without any genre labels |
| 4 | Genre diversity and exploration | clustering result | more genres? genres the user never chooses organically? |
| 5 | Session coherence | baskets (as notebook 02) | do recommended sessions hang together more tightly in sound? |
| 6 | User taste similarity in audio space | similarity (Week 4 idea, dense vectors) | are users' taste centroids more alike? |

Same fairness rules as notebook 02: real listens only, **distinct tracks** (organic listening is mostly replays), the same users in
both streams, equal-sized samples, associations only.

*Prerequisite:* `python scripts/extract_embeddings_50m.py` has been run (creates `data/processed/emb50m_*.npy`).
Run time: about 5-8 minutes. Memory: about 6 GB.

In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from scipy import stats
from scipy.sparse import csr_matrix
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# ---------------- parameters (change here, then Run All) ----------------
REAL_PCT = 50            # real listen = at least this % played
N_TRACKS = 100           # distinct tracks per user and stream
SESSION_GAP_S = 30 * 60
K_GENRE = 32             # number of pseudo-genres (k-means clusters)
CAP = 15                 # session baskets: max distinct tracks per basket
MAX_BASKETS_PER_USER = 100
SEED = 42

rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
COL = {"organic": "#2a9d8f", "recommended": "#e76f51"}

ROOT = Path.cwd()
while not (ROOT / "data" / "raw" / "yambda").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Could not find data/raw/yambda above " + str(Path.cwd()))
    ROOT = ROOT.parent
DATA, PROC = ROOT / "data" / "raw" / "yambda", ROOT / "data" / "processed"
verdict = []


def add_verdict(method, metric, organic, recommended, higher_means):
    rec_higher = recommended > organic
    broader = rec_higher if higher_means == "broader" else not rec_higher
    verdict.append({"analysis": method, "metric": metric, "organic": organic, "recommended": recommended,
                    "relative difference": (recommended - organic) / abs(organic) if organic else np.nan,
                    "recommended is": "broader" if broader else "narrower"})


def rank_in_group(g):
    order = np.argsort(g, kind="stable"); gs = g[order]
    start = np.r_[0, np.flatnonzero(gs[1:] != gs[:-1]) + 1]
    sizes = np.diff(np.r_[start, len(gs)])
    rank = np.empty(len(g), dtype=np.int64)
    rank[order] = np.arange(len(gs)) - np.repeat(start, sizes)
    return rank


def paired(org, rec):
    """Paired comparison over users: medians, bootstrap CI of the median difference, share, Wilcoxon p."""
    d = rec - org
    boot = np.median(d[rng.integers(0, len(d), (1000, len(d)))], axis=1)
    return dict(users=len(d), organic=np.median(org), recommended=np.median(rec), diff=np.median(d),
                lo=np.quantile(boot, 0.025), hi=np.quantile(boot, 0.975),
                share_rec_higher=(d > 0).mean(), p=stats.wilcoxon(d).pvalue)

print("project root:", ROOT)

## 0. Load listens, artists, albums, sessions and embeddings

In [ ]:
t0 = time.time()
df = pq.read_table(DATA / "flat/50m/multi_event.parquet",
                   columns=["uid", "item_id", "timestamp", "is_organic", "played_ratio_pct", "event_type"]).to_pandas()
df = df[df["event_type"] == "listen"].drop(columns="event_type").reset_index(drop=True)

emb_ids = np.load(PROC / "emb50m_item_ids.npy")
EN = np.load(PROC / "emb50m_norm.npy")                      # normalised embeddings (unit length), float32
L = int(max(df["item_id"].max(), emb_ids.max())) + 1
item_row = np.full(L, -1, dtype=np.int64); item_row[emb_ids] = np.arange(len(emb_ids))   # item id -> embedding row

def first_map(path, col):
    mp = pd.read_parquet(path).drop_duplicates("item_id")
    arr = np.full(L, -1, dtype=np.int32); ids = mp["item_id"].to_numpy(); ok = ids < L
    arr[ids[ok]] = mp[col].to_numpy()[ok]
    return arr
item_art = first_map(DATA / "artist_item_mapping.parquet", "artist_id")
item_alb = first_map(DATA / "album_item_mapping.parquet", "album_id")

item_arr = df["item_id"].to_numpy()
org_arr = df["is_organic"].to_numpy().astype(np.int8)
ratio_arr = df["played_ratio_pct"].to_numpy().astype(np.float32)
erow_arr = item_row[item_arr].astype(np.int32)
ts = df["timestamp"].to_numpy().astype(np.int64)
uid = df["uid"].to_numpy()
new_user = np.empty(len(uid), dtype=bool); new_user[0] = True; new_user[1:] = uid[1:] != uid[:-1]
uidx_arr = (np.cumsum(new_user) - 1).astype(np.int32)
n_users = int(uidx_arr[-1]) + 1
new_sess = np.empty(len(ts), dtype=bool); new_sess[0] = True
new_sess[1:] = new_user[1:] | ((ts[1:] - ts[:-1]) > SESSION_GAP_S)
sid_arr = (np.cumsum(new_sess) - 1).astype(np.int64)
del df

uniq = np.unique(uidx_arr.astype(np.int64) * (1 << 32) + item_arr)
item_users = np.bincount((uniq & 0xFFFFFFFF).astype(np.int64), minlength=L); del uniq
is_top = item_users >= np.quantile(item_users[item_users > 0], 0.99)
print(f"{len(ts):,} listens, {n_users:,} users; embeddings {EN.shape} for {len(emb_ids):,} tracks; "
      f"{(erow_arr >= 0).mean():.1%} of listens have an embedding ({time.time() - t0:.0f} s)")

## 1. Do the embeddings carry signal?
We have no genre labels, so we test the embeddings against the two content links we do have: tracks of the **same artist** and tracks of the
**same album** should be closer (higher cosine similarity) than random track pairs. Without this the whole analysis would be meaningless.

In [ ]:
def same_group_pairs(label, n_pairs=200_000):
    lab = label[emb_ids]
    order = np.argsort(lab, kind="stable"); ls = lab[order]
    same = (ls[1:] == ls[:-1]) & (ls[1:] >= 0)
    i1, i2 = order[:-1][same], order[1:][same]
    pick = rng.choice(len(i1), min(n_pairs, len(i1)), replace=False)
    return np.einsum("ij,ij->i", EN[i1[pick]], EN[i2[pick]])

cos_artist, cos_album = same_group_pairs(item_art), same_group_pairs(item_alb)
r1, r2 = rng.integers(0, len(EN), 200_000), rng.integers(0, len(EN), 200_000)
cos_rand = np.einsum("ij,ij->i", EN[r1], EN[r2])

def auc(pos, neg):
    r = stats.rankdata(np.r_[pos, neg]); return (r[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
bins = np.linspace(-0.4, 1, 80)
for name, v, c in [("random pair", cos_rand, "#6c757d"), ("same artist", cos_artist, "#264653"), ("same album", cos_album, "#e9c46a")]:
    ax[0].hist(v, bins=bins, histtype="step", density=True, lw=1.6, label=f"{name} (mean {v.mean():.2f})", color=c)
ax[0].set(title="Cosine similarity of track pairs", xlabel="cosine similarity", ylabel="density"); ax[0].legend()
sub = rng.choice(len(EN), 60_000, replace=False)
xy = PCA(n_components=2, random_state=SEED).fit_transform(EN[sub])
hb = ax[1].hexbin(xy[:, 0], xy[:, 1], C=np.log10(np.maximum(item_users[emb_ids[sub]], 1)), reduce_C_function=np.mean,
                  gridsize=45, cmap="viridis", mincnt=5)
plt.colorbar(hb, ax=ax[1], label="mean log10(users per track)")
ax[1].set(title="Map of the sound space (PCA of 60k tracks)", xlabel="PC1", ylabel="PC2")
plt.tight_layout(); plt.show()
print(f"AUC same-artist vs random: {auc(cos_artist, cos_rand):.3f}   same-album vs random: {auc(cos_album, cos_rand):.3f}   (0.5 = no signal)")

## 2. Audio spread per user
Same sample as notebook 02: for every user the first `N_TRACKS` distinct real tracks in each stream (users with enough tracks in both).
**Audio spread** = mean pairwise cosine *distance* between the user's tracks (higher = a wider range of sounds). It is computed
exactly from the sum vector: mean pairwise cosine = (|sum x|^2 - n) / (n (n - 1)).

In [ ]:
def build_sample(real_pct, n_tracks, drop_q=None):
    """Returns (users, items) with items of shape (U, 2, n_tracks); axis 1: 0 = recommended, 1 = organic."""
    m = (ratio_arr >= real_pct) & (erow_arr >= 0)
    if drop_q:        # popularity control: drop tracks above this quantile of "users per track"
        m &= item_users[item_arr] < np.quantile(item_users[item_users > 0], drop_q)
    ui, it, og = uidx_arr[m], item_arr[m], org_arr[m].astype(np.int64)
    _, fi = np.unique((ui.astype(np.int64) << 33) | (it.astype(np.int64) << 1) | og, return_index=True); fi.sort()
    ui, it, og = ui[fi], it[fi], og[fi]
    g = ui.astype(np.int64) * 2 + og
    rk = rank_in_group(g)
    cnt = np.bincount(g, minlength=2 * n_users).reshape(n_users, 2)
    elig = (cnt >= n_tracks).all(axis=1)
    sel = (rk < n_tracks) & elig[ui]
    gs, its = g[sel], it[sel]
    order = np.argsort(gs, kind="stable"); gs, its = gs[order], its[order]
    return gs[::n_tracks][::2] // 2, its.reshape(-1, 2, n_tracks)

def audio_metrics(items):
    U, _, n = items.shape
    rows = item_row[items]
    S = np.empty((U, 2, EN.shape[1]), dtype=np.float32)
    for s0 in range(0, U, 400):
        S[s0:s0 + 400] = EN[rows[s0:s0 + 400]].sum(axis=2)
    nrm2 = (S.astype(np.float64) ** 2).sum(-1)
    spread = 1 - (nrm2 - n) / (n * (n - 1))
    centroid = (S / np.sqrt(nrm2)[..., None]).astype(np.float32)
    return spread, centroid

users, items = build_sample(REAL_PCT, N_TRACKS)
spread, centroid = audio_metrics(items)
print(f"{len(users):,} users in the default sample")
configs = [("real>=30%, 100 tracks", 30, 100, None), ("real>=50%, 100 tracks (default)", 50, 100, None),
           ("real>=80%, 100 tracks", 80, 100, None), ("real>=50%, 50 tracks", 50, 50, None),
           ("real>=50%, 200 tracks", 50, 200, None), ("real>=50%, 100 tracks, top 1% popular tracks removed", 50, 100, 0.99),
           ("real>=50%, 100 tracks, top 20% popular tracks removed", 50, 100, 0.80)]
rows = []
for name, rp, nt, dt in configs:
    u_, it_ = (users, items) if name.endswith("(default)") else build_sample(rp, nt, dt)
    sp_ = spread if name.endswith("(default)") else audio_metrics(it_)[0]
    rows.append(dict(config=name, **paired(sp_[:, 1], sp_[:, 0])))
S2 = pd.DataFrame(rows).set_index("config")
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6), gridspec_kw={"width_ratios": [1, 1.1]})
d = spread[:, 0] - spread[:, 1]
ax[0].hist(d, bins=60, color="#6c757d"); ax[0].axvline(0, color="k", lw=1); ax[0].axvline(np.median(d), color="#c1121f", ls="--", lw=1.5)
ax[0].set(title="Audio spread: recommended - organic, per user", xlabel="difference in mean pairwise cosine distance")
y = np.arange(len(S2))[::-1]
ax[1].errorbar(S2["diff"], y, xerr=[S2["diff"] - S2["lo"], S2["hi"] - S2["diff"]], fmt="o", color="#264653", capsize=3)
ax[1].axvline(0, color="k", lw=1); ax[1].set_yticks(y); ax[1].set_yticklabels(S2.index, fontsize=8)
ax[1].set(title="Robustness (median difference, 95% CI)", xlabel="recommended - organic")
plt.tight_layout(); plt.show()
print(S2[["users", "organic", "recommended", "diff", "share_rec_higher", "p"]].to_string(float_format=lambda x: f"{x:.4g}"))
dflt = S2.loc[configs[1][0]]
add_verdict("2 audio spread", "mean pairwise audio distance (median)", dflt["organic"], dflt["recommended"], "broader")

## 3. Pseudo-genres: our own spherical k-means (Week 6)
Embeddings have unit length, so we cluster by cosine similarity: *spherical k-means* (assign each track to the centroid with the highest
dot product, then renormalise the centroid), started with k-means++ seeding. We fit on a random sample of tracks and then assign **all** tracks.
We check our implementation against scikit-learn, look at how the quality changes with k, and test whether the clusters are meaningful
(an artist's tracks should mostly fall in one pseudo-genre).

In [ ]:
def spherical_kmeans(X, k, seed, n_iter=60):
    r = np.random.default_rng(seed); n = len(X)
    C = np.empty((k, X.shape[1]), dtype=np.float32); C[0] = X[r.integers(n)]
    dist = 1 - X @ C[0]
    for j in range(1, k):                                   # k-means++ seeding with cosine distance
        pr = np.maximum(dist, 0) ** 2; C[j] = X[r.choice(n, p=pr / pr.sum())]
        dist = np.minimum(dist, 1 - X @ C[j])
    labels = np.full(n, -1)
    for _ in range(n_iter):
        new = np.argmax(X @ C.T, axis=1)
        if (new == labels).mean() > 0.9995:
            labels = new; break
        labels = new
        sums = csr_matrix((np.ones(n, np.float32), (labels, np.arange(n))), shape=(k, n)) @ X
        nrm = np.linalg.norm(sums, axis=1, keepdims=True)
        empty = nrm[:, 0] == 0
        sums[empty] = X[r.integers(n, size=empty.sum())]; nrm[empty] = 1
        C = (sums / np.maximum(nrm, 1e-9)).astype(np.float32)
    return C, labels, float((X * C[labels]).sum(1).mean())    # centroids, labels, mean cosine to own centroid

fit_idx = rng.choice(len(EN), 150_000, replace=False)
Xfit = EN[fit_idx]
sil_idx = rng.choice(len(Xfit), 10_000, replace=False)
rows = []
for k in (8, 16, 32, 64, 128):
    t1 = time.time(); C_, lab_, obj = spherical_kmeans(Xfit, k, SEED)
    rows.append({"k": k, "mean cosine to centroid": obj, "silhouette (10k sample)": silhouette_score(Xfit[sil_idx], lab_[sil_idx]),
                 "smallest cluster share": np.bincount(lab_, minlength=k).min() / len(lab_), "seconds": time.time() - t1})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.3f}"))

# check against scikit-learn on the same data. scikit-learn minimises Euclidean distance with un-normalised centroids, which is a
# different objective, so we compare like with like: the cosine objective of ITS partition (centroids renormalised) and the agreement.
from sklearn.metrics import adjusted_rand_score
C32, lab32, obj32 = spherical_kmeans(Xfit, 32, SEED)
sk = KMeans(32, n_init=1, random_state=SEED).fit(Xfit)
S_ = csr_matrix((np.ones(len(Xfit), np.float32), (sk.labels_, np.arange(len(Xfit)))), shape=(32, len(Xfit))) @ Xfit
C_sk = S_ / np.linalg.norm(S_, axis=1, keepdims=True)
seeds = [spherical_kmeans(Xfit, 32, s_)[2] for s_ in (1, 2, 3)]
print(f"mean cosine to centroid, k=32: ours {obj32:.4f} (other seeds {np.round(seeds, 4)}), "
      f"scikit-learn partition {float((Xfit * C_sk[sk.labels_]).sum(1).mean()):.4f}; "
      f"adjusted Rand index between the two partitions: {adjusted_rand_score(lab32, sk.labels_):.2f}")

In [ ]:
def fit_genres(k):
    C_, _, _ = spherical_kmeans(Xfit, k, SEED)
    lab = np.empty(len(EN), dtype=np.int16)
    for s0 in range(0, len(EN), 200_000):
        lab[s0:s0 + 200_000] = np.argmax(EN[s0:s0 + 200_000] @ C_.T, axis=1)
    return lab

genre_row = fit_genres(K_GENRE)                 # pseudo-genre of every embedded track (indexed by embedding row)

# do clusters mean something? artist purity: share of an artist's tracks in the artist's most common pseudo-genre
art_of_row = item_art[emb_ids]
t = pd.DataFrame({"artist": art_of_row, "genre": genre_row})
t = t[t["artist"] >= 0]
sizes = t.groupby("artist").size(); big = sizes[sizes >= 5].index
tb = t[t["artist"].isin(big)]
purity = tb.groupby(["artist", "genre"]).size().groupby("artist").max() / tb.groupby("artist").size()
shuf = tb.assign(genre=rng.permutation(tb["genre"].to_numpy()))
purity_shuf = shuf.groupby(["artist", "genre"]).size().groupby("artist").max() / shuf.groupby("artist").size()
print(f"artist purity (artists with >= 5 tracks, n={len(big):,}): mean {purity.mean():.2f}  vs  shuffled labels {purity_shuf.mean():.2f}")
cs = np.bincount(genre_row, minlength=K_GENRE)
print("pseudo-genre sizes (tracks): min {:,}  median {:,.0f}  max {:,}".format(cs.min(), np.median(cs), cs.max()))

## 4. Genre diversity and exploration
From the default sample (same users, same tracks as section 2) we now count, per user and stream:

* **distinct pseudo-genres** and **genre entropy** (higher = broader),
* **novel-genre share**: the share of a stream's tracks that fall in genres the user has *no* track in the *other* stream.
  For the recommended stream this is "exploration beyond what the user chooses"; for the organic stream it is the reverse and serves as the baseline.

In [ ]:
def genre_counts(items, genre_of_row):
    U, _, n = items.shape
    G = genre_of_row[item_row[items]].astype(np.int64)
    K = int(genre_of_row.max()) + 1
    idx = (np.arange(U)[:, None, None] * 2 + np.arange(2)[None, :, None]) * K + G
    return np.bincount(idx.ravel(), minlength=U * 2 * K).reshape(U, 2, K)

def genre_metrics(items, genre_of_row):
    cnt = genre_counts(items, genre_of_row); n = items.shape[2]
    p = cnt / n
    ent = -np.where(p > 0, p * np.log2(np.where(p > 0, p, 1)), 0).sum(-1)
    novel_rec = (cnt[:, 0] * (cnt[:, 1] == 0)).sum(-1) / n      # recommended tracks in genres absent from the user's organic sample
    novel_org = (cnt[:, 1] * (cnt[:, 0] == 0)).sum(-1) / n
    return (cnt > 0).sum(-1), ent, novel_rec, novel_org, cnt

dg, ent, nov_rec, nov_org, gcnt = genre_metrics(items, genre_row)
res4 = {"distinct pseudo-genres": paired(dg[:, 1], dg[:, 0]), "genre entropy (bits)": paired(ent[:, 1], ent[:, 0]),
        "novel-genre share (recommended vs organic baseline)": paired(nov_org, nov_rec)}
print(pd.DataFrame(res4).T[["users", "organic", "recommended", "diff", "lo", "hi", "share_rec_higher", "p"]].to_string(float_format=lambda x: f"{x:.4g}"))

# robustness to the number of pseudo-genres
rows = []
for k in (16, 32, 64):
    gl = genre_row if k == K_GENRE else fit_genres(k)
    d_, e_, nr_, no_, _ = genre_metrics(items, gl)
    rows.append({"k": k, "distinct genres: organic": np.median(d_[:, 1]), "recommended": np.median(d_[:, 0]),
                 "users rec > org (distinct)": (d_[:, 0] > d_[:, 1]).mean(), "median entropy diff": np.median(e_[:, 0] - e_[:, 1]),
                 "novel share organic": np.median(no_), "novel share recommended": np.median(nr_)})
print("\nRobustness to k:"); print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.3f}"))

# which genres does each stream over-represent?
share = gcnt.sum(0) / (gcnt.shape[0] * N_TRACKS)                 # (2, K): rows 0 recommended, 1 organic
lr = np.log2(share[0] / share[1]); order = np.argsort(lr)
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
ax[0].bar(range(K_GENRE), lr[order], color=np.where(lr[order] > 0, COL["recommended"], COL["organic"]))
ax[0].set_xticks(range(K_GENRE)); ax[0].set_xticklabels(order, fontsize=7)
ax[0].axhline(0, color="k", lw=1)
ax[0].set(title="Pseudo-genre share: log2(recommended / organic)", xlabel="pseudo-genre id", ylabel="log2 ratio")
ax[1].scatter(share[1] * 100, share[0] * 100, color="#264653")
mx = max(share.max() * 100, 1); ax[1].plot([0, mx], [0, mx], "k:", lw=1)
for g_ in np.argsort(np.abs(lr))[::-1][:5]:
    ax[1].annotate(str(g_), (share[1, g_] * 100, share[0, g_] * 100), fontsize=8)
ax[1].set(title="Share of tracks per pseudo-genre", xlabel="organic %", ylabel="recommended %")
plt.tight_layout(); plt.show()
add_verdict("4 pseudo-genres", "distinct pseudo-genres (median)", res4["distinct pseudo-genres"]["organic"], res4["distinct pseudo-genres"]["recommended"], "broader")
add_verdict("4 pseudo-genres", "genre entropy (median)", res4["genre entropy (bits)"]["organic"], res4["genre entropy (bits)"]["recommended"], "broader")
add_verdict("4 pseudo-genres", "novel-genre share", np.median(nov_org), np.median(nov_rec), "broader")

## 5. Session coherence
Baskets as in notebook 02, but built from distinct **tracks** (first `CAP` distinct real tracks of a user in one session and stream),
with the **same number of baskets of every size** in both streams. For each basket:

* **coherence** = mean pairwise cosine similarity of its tracks (higher = the session stays within one sound),
* **distinct pseudo-genres per track** (lower = more homogeneous).

We average per user and compare the two streams within users. If recommended sessions are more *templated*, coherence should be higher.

In [ ]:
elig_u = np.zeros(n_users, dtype=bool); elig_u[users] = True
m = (ratio_arr >= REAL_PCT) & (erow_arr >= 0) & elig_u[uidx_arr]
bs, bo, bi_, bu = sid_arr[m], org_arr[m].astype(np.int64), item_arr[m].astype(np.int64), uidx_arr[m]
_, fi = np.unique(((bs * 2 + bo) << 24) | bi_, return_index=True); fi.sort()
bs, bo, bi_, bu = bs[fi], bo[fi], bi_[fi], bu[fi]
gb = bs * 2 + bo
keep = rank_in_group(gb) < CAP
gb, bi_, bu, bo = gb[keep], bi_[keep], bu[keep], bo[keep]
order = np.argsort(gb, kind="stable"); gb, bi_, bu, bo = gb[order], bi_[order], bu[order], bo[order]
start = np.r_[0, np.flatnonzero(gb[1:] != gb[:-1]) + 1]
size = np.diff(np.r_[start, len(gb)]); b_user, b_org = bu[start], bo[start]

cand = np.flatnonzero(size >= 2); cand = cand[rng.permutation(len(cand))]
cand = cand[rank_in_group(b_user[cand] * 2 + b_org[cand]) < MAX_BASKETS_PER_USER]
chosen = []
for k in range(2, CAP + 1):
    ik = cand[size[cand] == k]; o = ik[b_org[ik] == 1]; rc = ik[b_org[ik] == 0]
    n_ = min(len(o), len(rc))
    if n_:
        chosen.append((k, rng.choice(o, n_, replace=False), rng.choice(rc, n_, replace=False)))

recs = []
for k, o, rc in chosen:
    for s, idxs in (("organic", o), ("recommended", rc)):
        rows_ = item_row[bi_[start[idxs][:, None] + np.arange(k)]]
        Sv = EN[rows_].sum(axis=1).astype(np.float64)
        coh = ((Sv ** 2).sum(1) - k) / (k * (k - 1))
        Gs = np.sort(genre_row[rows_], axis=1)
        dist_g = (1 + (np.diff(Gs, axis=1) != 0).sum(1)) / k
        recs.append(pd.DataFrame({"stream": s, "size": k, "user": b_user[idxs], "coherence": coh, "genres per track": dist_g}))
BK = pd.concat(recs, ignore_index=True)
print("baskets per stream:", BK.groupby("stream").size().to_dict(), "| size distributions identical:",
      bool((BK[BK.stream == "organic"]["size"].value_counts().sort_index() == BK[BK.stream == "recommended"]["size"].value_counts().sort_index()).all()))
per_user = BK.groupby(["user", "stream"])[["coherence", "genres per track"]].mean().unstack("stream")
cnt_u = BK.groupby(["user", "stream"]).size().unstack()
ok_users = cnt_u.index[(cnt_u >= 5).all(axis=1)]
pu = per_user.loc[ok_users]
res5 = {"session coherence": paired(pu[("coherence", "organic")].to_numpy(), pu[("coherence", "recommended")].to_numpy()),
        "distinct genres per track": paired(pu[("genres per track", "organic")].to_numpy(), pu[("genres per track", "recommended")].to_numpy())}
print(f"users with >= 5 matched baskets in both streams: {len(ok_users):,}")
print(pd.DataFrame(res5).T[["users", "organic", "recommended", "diff", "lo", "hi", "share_rec_higher", "p"]].to_string(float_format=lambda x: f"{x:.4g}"))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
for s in ("organic", "recommended"):
    g_ = BK[BK.stream == s].groupby("size")["coherence"].mean()
    ax[0].plot(g_.index, g_.values, "o-", color=COL[s], label=s)
ax[0].set(title="Session coherence by basket size", xlabel="distinct tracks in the basket", ylabel="mean pairwise cosine"); ax[0].legend()
d = pu[("coherence", "recommended")] - pu[("coherence", "organic")]
ax[1].hist(d, bins=50, color="#6c757d"); ax[1].axvline(0, color="k", lw=1); ax[1].axvline(d.median(), color="#c1121f", ls="--", lw=1.5)
ax[1].set(title="Per user: coherence recommended - organic", xlabel="difference")
plt.tight_layout(); plt.show()
add_verdict("5 sessions", "session coherence (median)", res5["session coherence"]["organic"], res5["session coherence"]["recommended"], "narrower")
add_verdict("5 sessions", "distinct genres per track (median)", res5["distinct genres per track"]["organic"], res5["distinct genres per track"]["recommended"], "broader")

## 6. Are users more alike in audio space?
Notebook 02 asked this with Jaccard on artist sets. With embeddings every user has a **taste centroid** per stream (the mean of their
`N_TRACKS` track vectors). We compare the cosine similarity between users' centroids: if recommendations make users alike, the recommended
centroids should be closer to each other than the organic ones. This has no set-size problem, because every centroid is built from the same number of tracks.

In [ ]:
Co, Cr = centroid[:, 1], centroid[:, 0]
iu = np.triu_indices(len(users), 1)
sim = {"organic": Co @ Co.T, "recommended": Cr @ Cr.T}
vals = {s: sim[s][iu] for s in sim}
rows = []
for s in sim:
    v = vals[s]; M_ = sim[s].copy(); np.fill_diagonal(M_, -1)
    rows.append({"stream": s, "mean cosine": v.mean(), "p50": np.median(v), "p99": np.quantile(v, 0.99),
                 "share >= 0.9": (v >= 0.9).mean(), "share >= 0.95": (v >= 0.95).mean(), "median nearest-neighbour cosine": np.median(M_.max(axis=1))})
T6 = pd.DataFrame(rows).set_index("stream")
print(T6.to_string(float_format=lambda x: f"{x:.4f}"))
idx = rng.choice(len(vals["organic"]), 400_000, replace=False)
dd = vals["recommended"][idx] - vals["organic"][idx]
print(f"\nsame pair of users: recommended centroids more similar in {(dd > 0).mean():.1%} of pairs (mean difference {dd.mean():+.4f}, p = {stats.wilcoxon(dd).pvalue:.1e})")
nn = {s: (lambda M_: (np.fill_diagonal(M_, -1), M_.max(axis=1))[1])(sim[s].copy()) for s in sim}
res6 = paired(nn["organic"], nn["recommended"])
print(f"nearest-neighbour cosine per user: organic {res6['organic']:.4f}, recommended {res6['recommended']:.4f}, "
      f"{res6['share_rec_higher']:.1%} of users have a closer twin in the recommended stream (p = {res6['p']:.1e})")
# how far apart are a user's own two centroids, compared with two random users?
own = np.einsum("ij,ij->i", Co, Cr)
rand_pair = np.einsum("ij,ij->i", Co, Cr[rng.permutation(len(Cr))])
print(f"cosine between a user's own organic and recommended centroid: {np.median(own):.3f}; organic user vs random other user's recommended: {np.median(rand_pair):.3f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
for s in sim:
    ax[0].hist(vals[s], bins=np.linspace(0.3, 1, 80), histtype="step", color=COL[s], lw=1.6, label=f"{s} (mean {vals[s].mean():.3f})", density=True)
ax[0].set(yscale="log", title="Cosine similarity of user taste centroids", xlabel="cosine", ylabel="density"); ax[0].legend()
ax[1].hist(own, bins=40, alpha=0.8, color="#264653", label="own organic vs own recommended")
ax[1].hist(rand_pair, bins=40, alpha=0.6, color="#adb5bd", label="organic vs another user's recommended")
ax[1].set(title="Is a user's recommended taste close to their organic taste?", xlabel="cosine"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
add_verdict("6 taste centroids", "mean user-user cosine", T6.loc["organic", "mean cosine"], T6.loc["recommended", "mean cosine"], "narrower")
add_verdict("6 taste centroids", "median nearest-neighbour cosine", res6["organic"], res6["recommended"], "narrower")

## Putting it together

In [ ]:
V = pd.DataFrame(verdict)
pd.set_option("display.width", 200)
print(V.to_string(index=False, float_format=lambda x: f"{x:,.4f}"))
print()
print(V.groupby("analysis")["recommended is"].value_counts().unstack(fill_value=0).to_string())

## Conclusion (default parameters: real listen >= 50 %, 100 distinct tracks per stream, k = 32 pseudo-genres)

**In sound space the recommended stream is narrower, not broader.** This differs from the artist-level result of notebook 02
(more distinct artists), so the full answer has two sides.

| Analysis | Organic | Recommended | Reading |
|---|---|---|---|
| Audio spread per user (mean pairwise cosine distance) | 0.698 | 0.658 | narrower; lower for 68 % of users |
| Distinct pseudo-genres per 100 tracks | 15 | 13 | narrower |
| Genre entropy (bits) | 2.92 | 2.72 | narrower |
| Tracks in genres the user never picks in the other stream | 10 % | 6 % | recommendations stay inside the user's own territory |
| Session coherence (mean pairwise cosine within a session) | 0.477 | 0.417 | recommended sessions are *more varied* inside one session |
| Distinct genres per track within a session | 0.51 | 0.54 | same reading |
| Mean cosine between two users' taste centroids | 0.501 | 0.475 | users are less alike on average ... |
| Nearest-neighbour cosine (closest other user) | 0.936 | 0.956 | ... but 75 % of users have a closer "twin" |

**How the two notebooks fit together.** Recommended listening plays *more different artists* and *varies more within a session*, but
keeps them inside a *smaller region of the sound space* than what users choose themselves. A user's own organic and recommended
centroids are very close (cosine 0.89, against 0.47 for two random users), so the recommendations are strongly personalised.
Our reading is "broader in artists, narrower in sound", a hypothesis that fits all the numbers but is not proven by them.

**Robustness.** The spread difference has the same sign and a confidence interval that excludes zero in all seven configurations
(real-listen threshold 30/50/80 %, 50/100/200 tracks, top 1 % and top 20 % of popular tracks removed). The popularity confound
(popular tracks cluster in one region of the sound space and are played more in the recommended stream) therefore does not explain it,
but note that the top-20 % control keeps only 420 users. The genre results hold for k = 16, 32 and 64. Our k-means matches scikit-learn
(same mean cosine to the centroid, adjusted Rand index about 0.7). Exact numbers vary slightly between runs (for example 14 or 15 median
genres) because the clustering sample depends on the random state.

**Limits.** Associations only. Pseudo-genres are clusters of a CNN embedding, not real genres, and we cannot name them. "First N distinct
tracks" favours early listening. The novel-genre share is not symmetric by construction (organic listening is the larger, more habitual stream),
so read it together with the other rows. Yambda-50M only, and 3 % of listens have no embedding (equally in both streams).